# ⭐<font color='lightgreen'>03 Build Star Schema</font>
---

### <font color='lightgreen'>Librerías</font>

In [190]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# Configuración de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

print("✅ Librerías cargadas")

✅ Librerías cargadas


### <font color='lightgreen'>Configuración</font>

In [191]:
# Rutas
DATA_CLEAN = "../data/clean"
DATA_CURATED = "../data/curated"

# Crear carpeta de salida
Path(DATA_CURATED).mkdir(parents=True, exist_ok=True)

# Diccionario para dimensiones y hechos
dimensiones = {}
hechos = {}

print(f"📁 Datos limpios: {DATA_CLEAN}")
print(f"📁 Modelo estrella: {DATA_CURATED}")

📁 Datos limpios: ../data/clean
📁 Modelo estrella: ../data/curated


### <font color='lightgreen'>1. Cargar Datos Limpios</font>

In [192]:
print("\n" + "=" * 50)
print("CARGANDO DATOS LIMPIOS...")
print("=" * 50)

datos = {}

# Configuración de archivos y sus columnas de fecha
archivos_config = [
    ("compras", ["fecha_emision", "fecha_pago"]),
    ("consumo_energetico", ["fecha"]),
    ("encuestas", ["fecha_encuesta"]),
    ("eventos_rrhh", ["fecha"]),
    ("personal_nomina", ["mes", "fecha_ingreso"]),
    ("residuos", ["fecha_semana"]),
    ("ventas", ["fecha"])
]

for nombre, columnas_fecha in archivos_config:
    file_path = f"{DATA_CLEAN}/{nombre}_clean.csv"
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
        
        # Convertir columnas de fecha a datetime
        for col in columnas_fecha:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
        
        datos[nombre] = df
        print(f"✅ {nombre}: {len(df):,} filas, {len(df.columns)} columnas")
        for col in columnas_fecha:
            if col in df.columns:
                print(f"   → {col}: convertida a datetime")
    except Exception as e:
        print(f"❌ {nombre}: {e}")


CARGANDO DATOS LIMPIOS...
✅ compras: 864 filas, 8 columnas
   → fecha_emision: convertida a datetime
   → fecha_pago: convertida a datetime
✅ consumo_energetico: 178 filas, 13 columnas
   → fecha: convertida a datetime
✅ encuestas: 189 filas, 12 columnas
   → fecha_encuesta: convertida a datetime
✅ eventos_rrhh: 537 filas, 10 columnas
   → fecha: convertida a datetime
✅ personal_nomina: 1,378 filas, 12 columnas
   → mes: convertida a datetime
   → fecha_ingreso: convertida a datetime
✅ residuos: 2,508 filas, 11 columnas
   → fecha_semana: convertida a datetime
✅ ventas: 43,893 filas, 15 columnas
   → fecha: convertida a datetime


### <font color='lightgreen'>2. Crear Dimensiones</font>

#### <font color='lightgreen'>>> 2.1 Dimensión Tiempo</font>

In [193]:
print("\n" + "=" * 50)
print("CREANDO DIMENSIÓN TIEMPO...")
print("=" * 50)

# Recolectar todas las fechas de todos los datasets
todas_fechas = []

# Fechas de compras
if 'fecha_emision' in datos['compras'].columns:
    todas_fechas.extend(datos['compras']['fecha_emision'].dropna().tolist())

# Fechas de consumo energético
if 'fecha' in datos['consumo_energetico'].columns:
    todas_fechas.extend(datos['consumo_energetico']['fecha'].dropna().tolist())

# Fechas de encuestas
if 'fecha_encuesta' in datos['encuestas'].columns:
    todas_fechas.extend(datos['encuestas']['fecha_encuesta'].dropna().tolist())

# Fechas de eventos RRHH
if 'fecha' in datos['eventos_rrhh'].columns:
    todas_fechas.extend(datos['eventos_rrhh']['fecha'].dropna().tolist())

# Fechas de nómina
if 'mes' in datos['personal_nomina'].columns:
    todas_fechas.extend(datos['personal_nomina']['mes'].dropna().tolist())

# Fechas de residuos
if 'fecha_semana' in datos['residuos'].columns:
    todas_fechas.extend(datos['residuos']['fecha_semana'].dropna().tolist())

# Fechas de ventas
if 'fecha' in datos['ventas'].columns:
    todas_fechas.extend(datos['ventas']['fecha'].dropna().tolist())

# Convertir a set y ordenar (ya son datetime)
fechas_unicas = sorted(set(todas_fechas))

# Crear dataframe
dim_tiempo = pd.DataFrame({'fecha': fechas_unicas})

# Extraer componentes
dim_tiempo['id_tiempo'] = dim_tiempo['fecha'].dt.strftime('%Y%m%d').astype(int)
dim_tiempo['anio'] = dim_tiempo['fecha'].dt.year
dim_tiempo['mes'] = dim_tiempo['fecha'].dt.month
dim_tiempo['dia'] = dim_tiempo['fecha'].dt.day
dim_tiempo['trimestre'] = dim_tiempo['fecha'].dt.quarter
dim_tiempo['semana'] = dim_tiempo['fecha'].dt.isocalendar().week
dim_tiempo['nombre_mes'] = dim_tiempo['fecha'].dt.strftime('%B')
dim_tiempo['dia_semana'] = dim_tiempo['fecha'].dt.day_name()
dim_tiempo['es_finde'] = (dim_tiempo['fecha'].dt.dayofweek >= 5).astype(int)

# Reordenar columnas para coincidir con el schema
dim_tiempo = dim_tiempo[['id_tiempo', 'fecha', 'anio', 'mes', 'dia', 'trimestre', 
                         'semana', 'nombre_mes', 'dia_semana', 'es_finde']]

print(f"✅ dim_tiempo: {len(dim_tiempo)} fechas únicas")
if len(dim_tiempo) > 0:
    print(f"   📅 Rango: {dim_tiempo['fecha'].min().date()} → {dim_tiempo['fecha'].max().date()}")
    print(f"   📊 Años: {sorted(dim_tiempo['anio'].unique())}")

dimensiones['dim_tiempo'] = dim_tiempo


CREANDO DIMENSIÓN TIEMPO...
✅ dim_tiempo: 2558 fechas únicas
   📅 Rango: 2018-01-01 → 2025-12-31
   📊 Años: [np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


#### <font color='lightgreen'>>> 2.2 Dimensión Area</font>

In [194]:
print("\n" + "=" * 50)
print("CREANDO DIMENSIÓN ÁREA...")
print("=" * 50)

# Recolectar áreas de diferentes fuentes
areas = set()

# Desde nómina
if 'area' in datos['personal_nomina'].columns:
    areas.update(datos['personal_nomina']['area'].dropna().unique())

# Desde encuestas
if 'area' in datos['encuestas'].columns:
    areas.update(datos['encuestas']['area'].dropna().unique())

# Desde eventos RRHH
if 'area' in datos['eventos_rrhh'].columns:
    areas.update(datos['eventos_rrhh']['area'].dropna().unique())

# Crear dataframe de áreas
dim_area = pd.DataFrame(sorted(areas), columns=['nombre_area'])
dim_area.insert(0, 'id_area', range(1, len(dim_area) + 1))

print(f"✅ dim_area: {len(dim_area)} áreas únicas")
print(f"   📍 Áreas: {list(dim_area['nombre_area'])}")

dimensiones['dim_area'] = dim_area


CREANDO DIMENSIÓN ÁREA...
✅ dim_area: 5 áreas únicas
   📍 Áreas: ['Administración', 'Logística', 'Taller', 'Todas', 'Ventas']


#### <font color='lightgreen'>>> 2.3 Dimensión Empleado</font>

In [195]:
print("\n" + "=" * 50)
print("CREANDO DIMENSIÓN EMPLEADO...")
print("=" * 50)

# Crear dimensión empleado desde nómina
if 'personal_nomina' in datos:
    df_nomina = datos['personal_nomina']
    
    # Seleccionar columnas relevantes
    columnas_empleado = ['id_empleado', 'nombre', 'area', 'genero', 'fecha_ingreso']
    columnas_existentes = [col for col in columnas_empleado if col in df_nomina.columns]
    
    dim_empleado = df_nomina[columnas_existentes].drop_duplicates(subset=['id_empleado']).copy()
    dim_empleado = dim_empleado.sort_values('id_empleado').reset_index(drop=True)
    
    print(f"✅ dim_empleado: {len(dim_empleado)} empleados únicos")
    print(f"   👥 Muestra: {dim_empleado.head(3).to_dict('records')}")
    
    dimensiones['dim_empleado'] = dim_empleado
else:
    print("⚠️  No se encontró datos de nómina para crear dim_empleado")


CREANDO DIMENSIÓN EMPLEADO...
✅ dim_empleado: 25 empleados únicos
   👥 Muestra: [{'id_empleado': 'E001', 'nombre': 'Fernández, Carlos', 'area': 'Administración', 'genero': 'M', 'fecha_ingreso': Timestamp('2018-01-01 00:00:00')}, {'id_empleado': 'E002', 'nombre': 'López, María', 'area': 'Ventas', 'genero': 'F', 'fecha_ingreso': Timestamp('2018-01-01 00:00:00')}, {'id_empleado': 'E003', 'nombre': 'García, Juan', 'area': 'Ventas', 'genero': 'M', 'fecha_ingreso': Timestamp('2018-01-01 00:00:00')}]


#### <font color='lightgreen'>>> 2.4 Dimensión Proveedor (desde compras)</font>

In [196]:
print("\n" + "=" * 50)
print("CREANDO DIMENSIÓN PROVEEDOR...")
print("=" * 50)

if 'compras' in datos:
    df_compras = datos['compras']
    
    if 'proveedor' in df_compras.columns:
        # Extraer proveedores únicos
        dim_proveedor = pd.DataFrame(
            df_compras['proveedor'].dropna().unique(),
            columns=['nombre_proveedor']
        )
        dim_proveedor.insert(0, 'id_proveedor', range(1, len(dim_proveedor) + 1))
        
        print(f"✅ dim_proveedor: {len(dim_proveedor)} proveedores únicos")
        print(f"   🏢 Muestra: {list(dim_proveedor['nombre_proveedor'].head(5))}")
        
        dimensiones['dim_proveedor'] = dim_proveedor
    else:
        print("⚠️  Columna 'proveedor' no encontrada en compras")
else:
    print("⚠️  Datos de compras no disponibles")


CREANDO DIMENSIÓN PROVEEDOR...
✅ dim_proveedor: 5 proveedores únicos
   🏢 Muestra: ['DistribuidoraTech SA', 'MegaComputo SRL', 'ElectroImport SA', 'AccesoriosTotal SRL', 'NetSupply SA']


#### <font color='lightgreen'>>> 2.5 Dimensión Categoría (productos)</font>

In [197]:
print("\n" + "=" * 50)
print("CREANDO DIMENSIÓN CATEGORÍA...")
print("=" * 50)

if 'ventas' in datos:
    df_ventas = datos['ventas']
    
    if 'categoria' in df_ventas.columns:
        dim_categoria = pd.DataFrame(
            df_ventas['categoria'].dropna().unique(),
            columns=['nombre_categoria']
        )
        dim_categoria.insert(0, 'id_categoria', range(1, len(dim_categoria) + 1))
        
        print(f"✅ dim_categoria: {len(dim_categoria)} categorías únicas")
        print(f"   📦 Categorías: {list(dim_categoria['nombre_categoria'])}")
        
        dimensiones['dim_categoria'] = dim_categoria
    else:
        print("⚠️  Columna 'categoria' no encontrada en ventas")
else:
    print("⚠️  Datos de ventas no disponibles")


CREANDO DIMENSIÓN CATEGORÍA...
✅ dim_categoria: 10 categorías únicas
   📦 Categorías: ['Periféricos', 'Smartphones', 'Seguridad', 'TV/Audio', 'Accesorios', 'Computación', 'Tablets', 'Almacenamiento', 'Redes', 'Servicios']


#### <font color='lightgreen'>>> 2.6 Dimensión Métrica (catálogo)</font>

In [198]:
print("\n" + "=" * 50)
print("CREANDO DIMENSIÓN MÉTRICA...")
print("=" * 50)

# Catálogo de métricas según definición del proyecto
metricas = [
    {'id_metrica': 1, 'nombre': 'Ingresos', 'categoria': 'Financiera', 'subcategoria': 'Ventas', 'unidad': 'ARS', 'descripcion': 'Suma de todas las ventas'},
    {'id_metrica': 2, 'nombre': 'Costo Compras', 'categoria': 'Financiera', 'subcategoria': 'Insumos', 'unidad': 'ARS', 'descripcion': 'Total de compras a proveedores'},
    {'id_metrica': 3, 'nombre': 'Gasto Personal', 'categoria': 'Financiera', 'subcategoria': 'RRHH', 'unidad': 'ARS', 'descripcion': 'Suma de salarios y bonos'},
    {'id_metrica': 4, 'nombre': 'Consumo Energético', 'categoria': 'E', 'subcategoria': 'Energía', 'unidad': 'kWh', 'descripcion': 'Consumo total de energía'},
    {'id_metrica': 5, 'nombre': 'Tasa Reciclaje', 'categoria': 'E', 'subcategoria': 'Residuos', 'unidad': '%', 'descripcion': 'Porcentaje de residuos reciclados'},
    {'id_metrica': 6, 'nombre': 'Satisfacción Laboral', 'categoria': 'S', 'subcategoria': 'Clima', 'unidad': 'puntos', 'descripcion': 'Promedio de satisfacción general'},
    {'id_metrica': 7, 'nombre': 'Horas Capacitación', 'categoria': 'S', 'subcategoria': 'Desarrollo', 'unidad': 'horas', 'descripcion': 'Total de horas de formación'},
    {'id_metrica': 8, 'nombre': 'Días Baja Accidentes', 'categoria': 'S', 'subcategoria': 'Seguridad', 'unidad': 'días', 'descripcion': 'Días de baja por accidentes'},
    {'id_metrica': 9, 'nombre': 'Intensidad Energética', 'categoria': 'E', 'subcategoria': 'Eficiencia', 'unidad': 'kWh/ARS', 'descripcion': 'Consumo energético por peso de ingreso', 'derivada': True},
    {'id_metrica': 10, 'nombre': 'Productividad por Empleado', 'categoria': 'Financiera', 'subcategoria': 'Productividad', 'unidad': 'ARS/empleado', 'descripcion': 'Ingresos por empleado', 'derivada': True},
]

dim_metrica = pd.DataFrame(metricas)
print(f"✅ dim_metrica: {len(dim_metrica)} métricas definidas")
print(f"   📊 Categorías: {dim_metrica['categoria'].value_counts().to_dict()}")

dimensiones['dim_metrica'] = dim_metrica


CREANDO DIMENSIÓN MÉTRICA...
✅ dim_metrica: 10 métricas definidas
   📊 Categorías: {'Financiera': 4, 'E': 3, 'S': 3}


### <font color='lightgreen'>3. Crear Tabla de Hechos (fact_monitoreo)</font>

In [199]:
# Lista para acumular todas las mediciones
mediciones = []
id_monitoreo_counter = 1

# Obtener diccionario de mapeo de fechas a id_tiempo
mapa_id_tiempo = dict(zip(dim_tiempo['fecha'], dim_tiempo['id_tiempo']))

In [200]:
# ------------------------------------------------------------
# 3.1 Métrica 1: Ingresos (desde ventas)
# ------------------------------------------------------------

if 'ventas' in datos and 'fecha' in datos['ventas'].columns and 'subtotal_ars' in datos['ventas'].columns:
    df = datos['ventas']
    ingresos = df.groupby(pd.Grouper(key='fecha', freq='ME')).agg({
        'subtotal_ars': 'sum'
    }).reset_index()
    
    for _, row in ingresos.iterrows():
        id_tiempo = mapa_id_tiempo.get(row['fecha'].replace(day=1))
        if id_tiempo:
            mediciones.append({
                'id_monitoreo': id_monitoreo_counter,
                'id_tiempo': id_tiempo,
                'id_metrica': 1,
                'id_area': 1,
                'valor': row['subtotal_ars'],
                'fuente': 'ventas'
            })
            id_monitoreo_counter += 1
    
    print(f"   ✅ Ingresos: {len(ingresos)} registros mensuales")

   ✅ Ingresos: 96 registros mensuales


In [201]:
# ------------------------------------------------------------
# 3.2 Métrica 2: Costo Compras
# ------------------------------------------------------------
if 'compras' in datos and 'fecha_emision' in datos['compras'].columns and 'monto_ars' in datos['compras'].columns:
    df = datos['compras']
    compras_agg = df.groupby(pd.Grouper(key='fecha_emision', freq='ME')).agg({
        'monto_ars': 'sum'
    }).reset_index()
    
    for _, row in compras_agg.iterrows():
        id_tiempo = mapa_id_tiempo.get(row['fecha_emision'].replace(day=1))
        if id_tiempo:
            mediciones.append({
                'id_monitoreo': id_monitoreo_counter,
                'id_tiempo': id_tiempo,
                'id_metrica': 2,
                'id_area': 1,
                'valor': row['monto_ars'],
                'fuente': 'compras'
            })
            id_monitoreo_counter += 1
    
    print(f"   ✅ Costo Compras: {len(compras_agg)} registros mensuales")

   ✅ Costo Compras: 96 registros mensuales


In [202]:
# ------------------------------------------------------------
# 3.3 Métrica 3: Gasto Personal
# ------------------------------------------------------------
if 'personal_nomina' in datos and 'mes' in datos['personal_nomina'].columns and 'total_devengado' in datos['personal_nomina'].columns:
    df = datos['personal_nomina']
    nomina_agg = df.groupby(pd.Grouper(key='mes', freq='ME')).agg({
        'total_devengado': 'sum'
    }).reset_index()
    
    for _, row in nomina_agg.iterrows():
        id_tiempo = mapa_id_tiempo.get(row['mes'].replace(day=1))
        if id_tiempo:
            mediciones.append({
                'id_monitoreo': id_monitoreo_counter,
                'id_tiempo': id_tiempo,
                'id_metrica': 3,
                'id_area': 1,
                'valor': row['total_devengado'],
                'fuente': 'personal_nomina'
            })
            id_monitoreo_counter += 1
    
    print(f"   ✅ Gasto Personal: {len(nomina_agg)} registros mensuales")

   ✅ Gasto Personal: 96 registros mensuales


In [203]:
# ------------------------------------------------------------
# 3.4 Métrica 4: Consumo Energético
# ------------------------------------------------------------
if 'consumo_energetico' in datos and 'fecha' in datos['consumo_energetico'].columns:
    df = datos['consumo_energetico']
    
    if 'kwh_totales' in df.columns:
        col_energia = 'kwh_totales'
    elif 'kwh_consumidos' in df.columns:
        col_energia = 'kwh_consumidos'
    else:
        col_energia = None
    
    if col_energia:
        energia_agg = df.groupby(pd.Grouper(key='fecha', freq='ME')).agg({
            col_energia: 'sum'
        }).reset_index()
        
        for _, row in energia_agg.iterrows():
            id_tiempo = mapa_id_tiempo.get(row['fecha'].replace(day=1))
            if id_tiempo:
                mediciones.append({
                    'id_monitoreo': id_monitoreo_counter,
                    'id_tiempo': id_tiempo,
                    'id_metrica': 4,
                    'id_area': 1,
                    'valor': row[col_energia],
                    'fuente': 'consumo_energetico'
                })
                id_monitoreo_counter += 1
        
        print(f"   ✅ Consumo Energético: {len(energia_agg)} registros mensuales")

   ✅ Consumo Energético: 96 registros mensuales


In [204]:
# ------------------------------------------------------------
# 3.5 Métrica 5: Tasa Reciclaje
# ------------------------------------------------------------
if 'residuos' in datos and 'fecha_semana' in datos['residuos'].columns and 'tasa_reciclaje' in datos['residuos'].columns:
    df = datos['residuos']
    reciclaje_agg = df.groupby(pd.Grouper(key='fecha_semana', freq='ME')).agg({
        'tasa_reciclaje': 'mean'
    }).reset_index()
    
    for _, row in reciclaje_agg.iterrows():
        id_tiempo = mapa_id_tiempo.get(row['fecha_semana'].replace(day=1))
        if id_tiempo:
            mediciones.append({
                'id_monitoreo': id_monitoreo_counter,
                'id_tiempo': id_tiempo,
                'id_metrica': 5,
                'id_area': 1,
                'valor': row['tasa_reciclaje'],
                'fuente': 'residuos'
            })
            id_monitoreo_counter += 1
    
    print(f"   ✅ Tasa Reciclaje: {len(reciclaje_agg)} registros mensuales")

   ✅ Tasa Reciclaje: 96 registros mensuales


In [205]:
# ------------------------------------------------------------
# 3.6 Métrica 6: Satisfacción Laboral
# ------------------------------------------------------------
if 'encuestas' in datos and 'fecha_encuesta' in datos['encuestas'].columns and 'satisfaccion_gral' in datos['encuestas'].columns:
    df = datos['encuestas']
    satisfaccion_agg = df.groupby(pd.Grouper(key='fecha_encuesta', freq='QE')).agg({
        'satisfaccion_gral': 'mean'
    }).reset_index()
    
    for _, row in satisfaccion_agg.iterrows():
        id_tiempo = mapa_id_tiempo.get(row['fecha_encuesta'].replace(day=1))
        if id_tiempo:
            mediciones.append({
                'id_monitoreo': id_monitoreo_counter,
                'id_tiempo': id_tiempo,
                'id_metrica': 6,
                'id_area': 1,
                'valor': row['satisfaccion_gral'],
                'fuente': 'encuestas'
            })
            id_monitoreo_counter += 1
    
    print(f"   ✅ Satisfacción Laboral: {len(satisfaccion_agg)} registros trimestrales")

   ✅ Satisfacción Laboral: 31 registros trimestrales


In [206]:
# ------------------------------------------------------------
# 3.7 Métrica 7: Horas Capacitación
# ------------------------------------------------------------
if 'eventos_rrhh' in datos and 'fecha' in datos['eventos_rrhh'].columns and 'horas_capacitacion' in datos['eventos_rrhh'].columns:
    df = datos['eventos_rrhh']
    df_capacitacion = df[df['tipo_evento'] == 'Capacitación']
    
    if len(df_capacitacion) > 0:
        horas_agg = df_capacitacion.groupby(pd.Grouper(key='fecha', freq='ME')).agg({
            'horas_capacitacion': 'sum'
        }).reset_index()
        
        for _, row in horas_agg.iterrows():
            id_tiempo = mapa_id_tiempo.get(row['fecha'].replace(day=1))
            if id_tiempo:
                mediciones.append({
                    'id_monitoreo': id_monitoreo_counter,
                    'id_tiempo': id_tiempo,
                    'id_metrica': 7,
                    'id_area': 1,
                    'valor': row['horas_capacitacion'],
                    'fuente': 'eventos_rrhh'
                })
                id_monitoreo_counter += 1
        
        print(f"   ✅ Horas Capacitación: {len(horas_agg)} registros mensuales")

   ✅ Horas Capacitación: 96 registros mensuales


In [207]:
# ------------------------------------------------------------
# 3.8 Métrica 8: Días Baja por Accidentes
# ------------------------------------------------------------
if 'eventos_rrhh' in datos and 'fecha' in datos['eventos_rrhh'].columns and 'dias_baja' in datos['eventos_rrhh'].columns:
    df = datos['eventos_rrhh']
    df_accidentes = df[df['tipo_evento'] == 'Accidente Laboral']
    
    if len(df_accidentes) > 0:
        dias_agg = df_accidentes.groupby(pd.Grouper(key='fecha', freq='ME')).agg({
            'dias_baja': 'sum'
        }).reset_index()
        
        for _, row in dias_agg.iterrows():
            id_tiempo = mapa_id_tiempo.get(row['fecha'].replace(day=1))
            if id_tiempo:
                mediciones.append({
                    'id_monitoreo': id_monitoreo_counter,
                    'id_tiempo': id_tiempo,
                    'id_metrica': 8,
                    'id_area': 1,
                    'valor': row['dias_baja'],
                    'fuente': 'eventos_rrhh'
                })
                id_monitoreo_counter += 1
        
        print(f"   ✅ Días Baja Accidentes: {len(dias_agg)} registros mensuales")



   ✅ Días Baja Accidentes: 96 registros mensuales


In [208]:
# Crear DataFrame de hechos
fact_monitoreo = pd.DataFrame(mediciones)

if not fact_monitoreo.empty:
    fact_monitoreo = fact_monitoreo.sort_values(['id_tiempo', 'id_metrica']).reset_index(drop=True)
    print(f"\n✅ fact_monitoreo: {len(fact_monitoreo)} registros")
    print(f"   📊 Métricas incluidas: {fact_monitoreo['id_metrica'].nunique()}")
    print(f"   📅 Períodos: {fact_monitoreo['id_tiempo'].nunique()}")
else:
    print("\n⚠️  No se generaron registros para fact_monitoreo")

hechos['fact_monitoreo'] = fact_monitoreo


✅ fact_monitoreo: 703 registros
   📊 Métricas incluidas: 8
   📅 Períodos: 96


### <font color='lightgreen'>4. Guardar Modelo Estrella</font>

In [209]:
print("\n" + "=" * 50)
print("GUARDANDO MODELO ESTRELLA...")
print("=" * 50)

# Guardar dimensiones
for nombre, df in dimensiones.items():
    output_file = f"{DATA_CURATED}/{nombre}.csv"
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"   💾 {nombre}.csv ({len(df)} filas)")

# Guardar tabla de hechos
for nombre, df in hechos.items():
    output_file = f"{DATA_CURATED}/{nombre}.csv"
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"   💾 {nombre}.csv ({len(df)} filas)")


GUARDANDO MODELO ESTRELLA...
   💾 dim_tiempo.csv (2558 filas)
   💾 dim_area.csv (5 filas)
   💾 dim_empleado.csv (25 filas)
   💾 dim_proveedor.csv (5 filas)
   💾 dim_categoria.csv (10 filas)
   💾 dim_metrica.csv (10 filas)
   💾 fact_monitoreo.csv (703 filas)


### <font color='lightgreen'>5. Resumen Final</font>

In [210]:
print("\n" + "=" * 50)
print("✅ MODELO ESTRELLA COMPLETADO")
print("=" * 50)

print("\n📊 Dimensiones creadas:")
for nombre, df in dimensiones.items():
    print(f"   • {nombre}: {len(df):,} registros")

print("\n📊 Tabla de Hechos:")
for nombre, df in hechos.items():
    print(f"   • {nombre}: {len(df):,} registros, {len(df.columns)} columnas")

# Esquema de relaciones
print("\n🔗 Esquema Estrella:")
print("""
                    ┌─────────────────┐
                    │   dim_tiempo    │
                    │   (fecha)       │
                    └────────┬────────┘
                             │
    ┌─────────────────┐      │      ┌─────────────────┐
    │   dim_area      ├──────┼──────┤   dim_empleado  │
    └─────────────────┘      │      └─────────────────┘
                             │
┌─────────────────┐          │          ┌─────────────────┐
│   dim_metrica   ├──────────┼──────────┤  dim_proveedor  │
└─────────────────┘          │          └─────────────────┘
                             │
                    ┌────────┴─────────┐
                    │  fact_mediciones │
                    │  (hechos)        │
                    └──────────────────┘
""")

print(f"\n📁 Archivos guardados en: {DATA_CURATED}/")
print("\n✅ Listo para conectar al dashboard (Power BI / Looker Studio)")


✅ MODELO ESTRELLA COMPLETADO

📊 Dimensiones creadas:
   • dim_tiempo: 2,558 registros
   • dim_area: 5 registros
   • dim_empleado: 25 registros
   • dim_proveedor: 5 registros
   • dim_categoria: 10 registros
   • dim_metrica: 10 registros

📊 Tabla de Hechos:
   • fact_monitoreo: 703 registros, 6 columnas

🔗 Esquema Estrella:

                    ┌─────────────────┐
                    │   dim_tiempo    │
                    │   (fecha)       │
                    └────────┬────────┘
                             │
    ┌─────────────────┐      │      ┌─────────────────┐
    │   dim_area      ├──────┼──────┤   dim_empleado  │
    └─────────────────┘      │      └─────────────────┘
                             │
┌─────────────────┐          │          ┌─────────────────┐
│   dim_metrica   ├──────────┼──────────┤  dim_proveedor  │
└─────────────────┘          │          └─────────────────┘
                             │
                    ┌────────┴─────────┐
                    │  fact